# Base de Datos ITER 2020 — Ciudad de México

Construcción de una base de datos combinada a partir de dos archivos del **Inventario Nacional de Viviendas (ITER) 2020** del INEGI para la Ciudad de México.

| Archivo | Contenido |
|---|---|
| Archivo 1 | Datos geográficos, demográficos y de empleo |
| Archivo 2 | Datos de servicios de salud y seguridad social |

## 1. Carga de datos

In [1]:
import pandas as pd

ruta1 = r"c:\Users\regin\OneDrive\Desktop\bases de datos\ITER2020 - 09 Ciudad de México (1).csv"
ruta2 = r"c:\Users\regin\OneDrive\Desktop\bases de datos\ITER2020 - 09 Ciudad de México (2).csv"

df1 = pd.read_csv(ruta1, encoding='latin1')
df2 = pd.read_csv(ruta2, encoding='latin1')

print(f"Archivo 1: {df1.shape[0]:,} filas | {df1.shape[1]} columnas")
print(f"Archivo 2: {df2.shape[0]:,} filas | {df2.shape[1]} columnas")

Archivo 1: 666 filas | 63 columnas
Archivo 2: 666 filas | 31 columnas


## 2. Construcción de la base combinada

Se conserva el **Archivo 1 completo** (geografía + demografía + empleo) y se agregan las columnas exclusivas de **servicios de salud** del Archivo 2.

La unión se realiza por `ENTIDAD`, `MUN` y `LOC`.

In [ ]:
# Columnas de salud: código INEGI → nombre descriptivo
cols_salud = {
    'PSINDER'    : 'Sin derechohabiencia',
    'PDER_SS'    : 'Con derechohabiencia',
    'PDER_IMSS'  : 'Derechohabiente IMSS',
    'PDER_ISTE'  : 'Derechohabiente ISSSTE federal',
    'PDER_ISTEE' : 'Derechohabiente ISSSTE estatal',
    'PAFIL_PDOM' : 'Afiliado Seguro Popular/Bienestar',
    'PDER_SEGP'  : 'Seguro Popular o Médico Siglo XXI',
    'PDER_IMSSB' : 'Derechohabiente IMSS-Bienestar',
    'PAFIL_IPRIV': 'Afiliado institución privada',
    'PAFIL_OTRAI': 'Afiliado otra institución',
}

df2_salud = df2[['ENTIDAD', 'MUN', 'LOC'] + list(cols_salud.keys())].rename(columns=cols_salud)

# Merge
df = pd.merge(df1, df2_salud, on=['ENTIDAD', 'MUN', 'LOC'], how='inner')

# Convertir columnas de edad a numérico para poder sumarlas
cols_edad = ['P_0A2','P_0A2_F','P_0A2_M',
             'P_3A5','P_3A5_F','P_3A5_M',
             'P_6A11','P_6A11_F','P_6A11_M',
             'P_12A14','P_12A14_F','P_12A14_M',
             'P_15A17','P_15A17_F','P_15A17_M',
             'P_18YMAS','P_18YMAS_F','P_18YMAS_M',
             'P_60YMAS','P_60YMAS_F','P_60YMAS_M']
df[cols_edad] = df[cols_edad].apply(pd.to_numeric, errors='coerce')

# ── Crear 4 grupos de edad ──────────────────────────────────────────────────
# Niños (0-11 años)
df['Niños']     = df['P_0A2']   + df['P_3A5']   + df['P_6A11']
df['Niños (F)'] = df['P_0A2_F'] + df['P_3A5_F'] + df['P_6A11_F']
df['Niños (M)'] = df['P_0A2_M'] + df['P_3A5_M'] + df['P_6A11_M']

# Jóvenes (12-17 años)
df['Jóvenes']     = df['P_12A14']   + df['P_15A17']
df['Jóvenes (F)'] = df['P_12A14_F'] + df['P_15A17_F']
df['Jóvenes (M)'] = df['P_12A14_M'] + df['P_15A17_M']

# Adultos (18-59 años) = 18 y más - 60 y más
df['Adultos']     = df['P_18YMAS']   - df['P_60YMAS']
df['Adultos (F)'] = df['P_18YMAS_F'] - df['P_60YMAS_F']
df['Adultos (M)'] = df['P_18YMAS_M'] - df['P_60YMAS_M']

# Tercera edad (60+ años)
df['Tercera edad']     = df['P_60YMAS']
df['Tercera edad (F)'] = df['P_60YMAS_F']
df['Tercera edad (M)'] = df['P_60YMAS_M']

# ── Eliminar columnas de edad individuales ──────────────────────────────────
cols_drop = ['P_0A2','P_0A2_F','P_0A2_M',
             'P_3A5','P_3A5_F','P_3A5_M',
             'P_6A11','P_6A11_F','P_6A11_M',
             'P_8A14','P_8A14_F','P_8A14_M',
             'P_12A14','P_12A14_F','P_12A14_M',
             'P_15A17','P_15A17_F','P_15A17_M',
             'P_18A24','P_18A24_F','P_18A24_M',
             'P_15A49_F',
             'P_60YMAS','P_60YMAS_F','P_60YMAS_M',
             'P_5YMAS','P_5YMAS_F','P_5YMAS_M',
             'P_12YMAS','P_12YMAS_F','P_12YMAS_M',
             'P_15YMAS','P_15YMAS_F','P_15YMAS_M',
             'POB0_14','POB15_64','POB65_MAS']
df.drop(columns=cols_drop, inplace=True)

# ── Renombrar el resto de columnas ──────────────────────────────────────────
nombres = {
    'ENTIDAD'  : 'Clave entidad',
    'NOM_ENT'  : 'Entidad',
    'MUN'      : 'Clave municipio',
    'NOM_MUN'  : 'Municipio',
    'LOC'      : 'Clave localidad',
    'NOM_LOC'  : 'Localidad',
    'LONGITUD' : 'Longitud',
    'LATITUD'  : 'Latitud',
    'ALTITUD'  : 'Altitud',
    'POBTOT'   : 'Población total',
    'POBFEM'   : 'Población femenina',
    'POBMAS'   : 'Población masculina',
    'REL_H_M'  : 'Relación hombres-mujeres',
    'P_3YMAS'  : 'Pob. 3 años y más',
    'P_3YMAS_F': 'Pob. 3 años y más (F)',
    'P_3YMAS_M': 'Pob. 3 años y más (M)',
    'P_18YMAS' : 'Pob. 18 años y más',
    'P_18YMAS_F': 'Pob. 18 años y más (F)',
    'P_18YMAS_M': 'Pob. 18 años y más (M)',
    'PEA'      : 'Pob. económicamente activa',
    'PEA_F'    : 'Pob. económicamente activa (F)',
    'PEA_M'    : 'Pob. económicamente activa (M)',
    'PE_INAC'  : 'Pob. económicamente inactiva',
    'PE_INAC_F': 'Pob. económicamente inactiva (F)',
    'PE_INAC_M': 'Pob. económicamente inactiva (M)',
    'POCUPADA' : 'Pob. ocupada',
}
df.rename(columns=nombres, inplace=True)

print(f"Base combinada: {df.shape[0]:,} filas | {df.shape[1]} columnas")
df.head()

## 3. Vista previa de la base combinada

In [3]:
df.head()

,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,LONGITUD,LATITUD,ALTITUD,POBTOT,...,PSINDER,PDER_SS,PDER_IMSS,PDER_ISTE,PDER_ISTEE,PAFIL_PDOM,PDER_SEGP,PDER_IMSSB,PAFIL_IPRIV,PAFIL_OTRAI
0,9,Ciudad de MÃ©xico,0,Total de la entidad Ciudad de MÃ©xico,0,Total de la Entidad,NaN,NaN,NaN,9209944,...,2502789,6689012,3881545,1128554,12484,104474,1203824,21158,444160,93084
1,9,Ciudad de MÃ©xico,0,Total de la entidad Ciudad de MÃ©xico,9998,Localidades de una vivienda,NaN,NaN,NaN,320,...,139,181,65,20,0,0,88,5,5,0
2,9,Ciudad de MÃ©xico,0,Total de la entidad Ciudad de MÃ©xico,9999,Localidades de dos viviendas,NaN,NaN,NaN,364,...,135,229,55,30,1,2,122,0,18,3
3,9,Ciudad de MÃ©xico,2,Azcapotzalco,0,Total del Municipio,NaN,NaN,NaN,432205,...,90370,341302,238847,48827,902,11600,34369,876,10983,2854
4,9,Ciudad de MÃ©xico,2,Azcapotzalco,1,Azcapotzalco,"99Â°11'03.698"" W","19Â°29'02.770"" N",2244.0,432205,...,90370,341302,238847,48827,902,11600,34369,876,10983,2854


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 73 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ENTIDAD      666 non-null    int64  
 1   NOM_ENT      666 non-null    object 
 2   MUN          666 non-null    int64  
 3   NOM_MUN      666 non-null    object 
 4   LOC          666 non-null    int64  
 5   NOM_LOC      666 non-null    object 
 6   LONGITUD     634 non-null    object 
 7   LATITUD      634 non-null    object 
 8   ALTITUD      634 non-null    float64
 9   POBTOT       666 non-null    int64  
 10  POBFEM       666 non-null    object 
 11  POBMAS       666 non-null    object 
 12  P_0A2        666 non-null    object 
 13  P_0A2_F      666 non-null    object 
 14  P_0A2_M      666 non-null    object 
 15  P_3YMAS      666 non-null    object 
 16  P_3YMAS_F    666 non-null    object 
 17  P_3YMAS_M    666 non-null    object 
 18  P_5YMAS      666 non-null    object 
 19  P_5YMAS_

## 4. Limpieza de datos

In [ ]:
df_limpio = df.copy()

# 1. Reemplazar cadenas "N/A" por NaN real
df_limpio.replace('N/A', pd.NA, inplace=True)

# 2. Eliminar filas de totales y resúmenes
filas_antes = len(df_limpio)
df_limpio = df_limpio[~df_limpio['Clave localidad'].isin([0, 9998, 9999])]
print(f"Filas eliminadas (totales/resúmenes) : {filas_antes - len(df_limpio)}")

# 3. Eliminar duplicados
filas_antes = len(df_limpio)
df_limpio.drop_duplicates(inplace=True)
print(f"Filas eliminadas (duplicados)        : {filas_antes - len(df_limpio)}")

# 4. Convertir columnas numéricas al tipo correcto
cols_texto = ['Entidad', 'Municipio', 'Localidad', 'Longitud', 'Latitud']
cols_numericas = df_limpio.columns.difference(cols_texto)
df_limpio[cols_numericas] = df_limpio[cols_numericas].apply(pd.to_numeric, errors='coerce')

# 5. Eliminar filas con valores nulos restantes
filas_antes = len(df_limpio)
df_limpio.dropna(inplace=True)
print(f"Filas eliminadas (valores nulos)     : {filas_antes - len(df_limpio)}")

# 6. Resetear índice
df_limpio.reset_index(drop=True, inplace=True)

print(f"\nFilas finales  : {df_limpio.shape[0]:,}")
print(f"Columnas       : {df_limpio.shape[1]}")
print(f"Valores nulos  : {df_limpio.isnull().sum().sum()}")
df_limpio.head()

## 5. Visualización de datos

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
colores = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle('ITER 2020 — Ciudad de México', fontsize=16, fontweight='bold', y=1.01)

# ── 1. Distribución de grupos de edad ───────────────────────────────────────
ax = axes[0, 0]
grupos = ['Niños\n(0-11)', 'Jóvenes\n(12-17)', 'Adultos\n(18-59)', 'Tercera edad\n(60+)']
cols   = ['Niños', 'Jóvenes', 'Adultos', 'Tercera edad']
totales = df_limpio[cols].sum()
bars = ax.bar(grupos, totales / 1e6, color=colores)
ax.bar_label(bars, labels=[f'{v/1e6:.2f}M' for v in totales], padding=4)
ax.set_title('Distribución por grupos de edad')
ax.set_ylabel('Millones de personas')
ax.set_ylim(0, totales.max() / 1e6 * 1.2)
ax.grid(axis='y', alpha=0.3)

# ── 2. Distribución por sexo en cada grupo de edad ──────────────────────────
ax = axes[0, 1]
x = range(len(grupos))
fem  = df_limpio[['Niños (F)', 'Jóvenes (F)', 'Adultos (F)', 'Tercera edad (F)']].sum() / 1e6
mas  = df_limpio[['Niños (M)', 'Jóvenes (M)', 'Adultos (M)', 'Tercera edad (M)']].sum() / 1e6
w = 0.35
b1 = ax.bar([i - w/2 for i in x], fem, w, label='Mujeres', color='#C44E52', alpha=0.85)
b2 = ax.bar([i + w/2 for i in x], mas, w, label='Hombres', color='#4C72B0', alpha=0.85)
ax.bar_label(b1, labels=[f'{v:.2f}M' for v in fem], padding=3, fontsize=9)
ax.bar_label(b2, labels=[f'{v:.2f}M' for v in mas], padding=3, fontsize=9)
ax.set_xticks(list(x)); ax.set_xticklabels(grupos)
ax.set_title('Grupos de edad por sexo')
ax.set_ylabel('Millones de personas')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# ── 3. Derechohabiencia (pastel) ─────────────────────────────────────────────
ax = axes[1, 0]
con_der = df_limpio['Con derechohabiencia'].sum()
sin_der = df_limpio['Sin derechohabiencia'].sum()
ax.pie([con_der, sin_der],
       labels=['Con derechohabiencia', 'Sin derechohabiencia'],
       autopct='%1.1f%%', colors=['#55A868', '#C44E52'],
       startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Cobertura de derechohabiencia')

# ── 4. Tipo de servicio de salud ─────────────────────────────────────────────
ax = axes[1, 1]
servicios = {
    'IMSS'              : df_limpio['Derechohabiente IMSS'].sum(),
    'Seguro Popular'    : df_limpio['Seguro Popular o Médico Siglo XXI'].sum(),
    'Inst. privada'     : df_limpio['Afiliado institución privada'].sum(),
    'ISSSTE federal'    : df_limpio['Derechohabiente ISSSTE federal'].sum(),
    'Seg. Pop/Bienestar': df_limpio['Afiliado Seguro Popular/Bienestar'].sum(),
    'IMSS-Bienestar'    : df_limpio['Derechohabiente IMSS-Bienestar'].sum(),
    'ISSSTE estatal'    : df_limpio['Derechohabiente ISSSTE estatal'].sum(),
    'Otra institución'  : df_limpio['Afiliado otra institución'].sum(),
}
serv_sorted = dict(sorted(servicios.items(), key=lambda x: x[1], reverse=True))
bars = ax.barh(list(serv_sorted.keys()), [v/1e6 for v in serv_sorted.values()],
               color='#4C72B0', alpha=0.85)
ax.bar_label(bars, labels=[f'{v/1e6:.2f}M' for v in serv_sorted.values()], padding=3)
ax.set_title('Población por tipo de servicio de salud')
ax.set_xlabel('Millones de personas')
ax.grid(axis='x', alpha=0.3)

# ── 5. Población por municipio (top 10) ──────────────────────────────────────
ax = axes[2, 0]
pob_mun = df_limpio.groupby('Municipio')['Población total'].sum().nlargest(10).sort_values()
bars = ax.barh(pob_mun.index, pob_mun.values / 1e6, color='#DD8452', alpha=0.85)
ax.bar_label(bars, labels=[f'{v/1e6:.2f}M' for v in pob_mun.values], padding=3)
ax.set_title('Top 10 municipios por población total')
ax.set_xlabel('Millones de personas')
ax.grid(axis='x', alpha=0.3)

# ── 6. Situación laboral ─────────────────────────────────────────────────────
ax = axes[2, 1]
pea    = df_limpio['Pob. económicamente activa'].sum()
inac   = df_limpio['Pob. económicamente inactiva'].sum()
ocup   = df_limpio['Pob. ocupada'].sum()
desoc  = pea - ocup
etiquetas = ['Ocupada', 'Desocupada', 'Econ. inactiva']
valores   = [ocup, desoc, inac]
colores2  = ['#55A868', '#C44E52', '#8C8C8C']
wedges, texts, autotexts = ax.pie(
    valores, labels=etiquetas, autopct='%1.1f%%',
    colors=colores2, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Situación laboral (Pob. 12 años y más)')

plt.tight_layout()
plt.savefig('graficas_iter2020.png', bbox_inches='tight', dpi=150)
plt.show()
print("Gráficas guardadas como graficas_iter2020.png")

## 4. Exportar a CSV

In [ ]:
df_limpio.to_csv('base_combinada.csv', index=False, encoding='utf-8-sig')
print("Archivo guardado como base_combinada.csv")